### <span style="color:#FFA726">Leave-Last-Out </span>

This notebook has three parts:

1. **Pre-Split Notes** — the plan, columns used, and sanity checks, before any code.
2. **Split Code** — sorting, carving out test/validation/train, saving to `datasets/`.
3. **Post-Split Sanity Checks** — verifying no leakage, no overlap, and correct counts.

### <span style="color:#FFA726">1. Pre-Split Notes</span>

**Goal:** For each user, hold out their most recent (latest timestamp) interaction 
as the test row, their second-to-last interaction as the validation row, and keep 
everything else as train.

**Columns involved:**
- Used for splitting/modeling: `user_id`, `parent_asin`, `rating`, `timestamp`
- `average_rating` must never enter the model (leakage risk) — interpretation/reporting only
- `title`, `categories`, `main_category` are for interpretation/reporting only, 
  not used in the split logic

**Steps:**

1. **Sort:** Sort `merged` by `user_id` first, then by `timestamp` within each user, 
   from oldest to newest.
2. **Carve out the test set:** For each user group, take the last row (i.e., the row 
   with the maximum timestamp per `user_id`) → this becomes the `test` set.
3. **Carve out the validation set:** After removing the test rows, for each user group 
   take the new last row (i.e., what was originally the second-to-last row per user) 
   → this becomes the `validation` set.
4. **Build the train set:** Remove both the test rows and the validation rows from 
   `merged` → everything remaining is `train`.
5. **Sanity checks:**
   - `test` should have exactly one row per user → total test rows = 657,203 
     (equal to the number of users)
   - `validation` should also have exactly one row per user → total validation 
     rows = 657,203
   - `train` + `validation` + `test` row counts should sum to exactly the original 
     `merged` row count
   - No user should end up with zero rows in `train` (since every user has a minimum 
     of 5 interactions, train should retain at least 3 rows per user — verified explicitly)

**Important note (from EDA findings):**
- Median interactions per user is **7** — a large share of users have very little 
  data (minimum 5)
- After the split, many users will be left with only **3 rows** in train (out of 5 
  interactions, 1 goes to test, 1 goes to validation)
- This creates a "sparse user" challenge for both kNN and ALS — an anticipated risk, 
  to be noted in the final report

### <span style="color:#FFA726">2. Split Code</span>

In [14]:
import pandas as pd

merged = pd.read_parquet('datasets/merged_movies_tv.parquet')
merged.head()

,user_id,parent_asin,rating,timestamp,title,categories,average_rating,main_category
0,AGXVBIUFLFGMVLATYXHJYL4A5Q7Q,B0002J58ME,5.0,1146713492000,10 Minute Solution: Pilates,"[Movies & TV, Featured Categories, DVD, Exerci...",4.6,Movies & TV
1,AGXVBIUFLFGMVLATYXHJYL4A5Q7Q,B0091VXC54,5.0,1443550066000,None,None,4.6,Prime Video
2,AGXVBIUFLFGMVLATYXHJYL4A5Q7Q,B016JGYSRY,5.0,1445968640000,None,None,4.6,Prime Video
3,AGXVBIUFLFGMVLATYXHJYL4A5Q7Q,B00OI7J214,5.0,1446579637000,None,None,4.6,Prime Video
4,AGXVBIUFLFGMVLATYXHJYL4A5Q7Q,B01AMUSHAC,5.0,1476708291000,None,None,4.8,Prime Video


In [15]:
merged = merged.sort_values(['user_id', 'timestamp'])

test = merged.groupby('user_id').tail(1)

remaining = merged.drop(test.index)
validation = remaining.groupby('user_id').tail(1)

train = remaining.drop(validation.index)

print("train:", len(train))
print("validation:", len(validation))
print("test:", len(test))

train: 6126723
validation: 657203
test: 657203


##### <span style="color:#BA68C8">Saving the split</span>

We save `train`, `validation`, and `test` to `datasets/` as parquet files, 
so we don't need to recompute the split later.

In [16]:
train.to_parquet('datasets/train.parquet', index=False)
validation.to_parquet('datasets/validation.parquet', index=False)
test.to_parquet('datasets/test.parquet', index=False)

print("Kaydedildi ✅")

Kaydedildi ✅


### <span style="color:#FFA726">3. Post-Split Sanity Checks</span>

Before trusting the split, we verify five things:

1. **No temporal leakage (per user):** `max(train timestamp) < validation timestamp < test timestamp` for every user.

*Why it matters:* confirms the model never trains on "future" data.

2. **No overlapping rows between sets:** the index sets of `train`, `validation`, and `test` are pairwise disjoint.

*Why it matters:* confirms no row is trained on and tested on at once.

3. **Row count conservation:** `len(train) + len(validation) + len(test) == len(merged)`.

*Why it matters:* confirms no rows were lost or duplicated.

4. **Minimum train coverage per user:** every user in `merged` has at least one row in `train`.

*Why it matters:* a user with no train history can't be modeled meaningfully.

5. **User count consistency:** `test`, `validation`, and the total unique user count in `merged` are all equal to 657,203.

*Why it matters:* confirms the leave-last-out logic applied correctly to every user.

In [17]:
# 1. No temporal leakage (per user)
train_max_ts = train.groupby('user_id')['timestamp'].max()
val_ts = validation.set_index('user_id')['timestamp']
test_ts = test.set_index('user_id')['timestamp']

leakage_check = (train_max_ts < val_ts) & (val_ts < test_ts)
print("1. Temporal leakage violations:", (~leakage_check).sum())

# 2. No overlapping rows between sets
overlap_train_val = train.index.intersection(validation.index)
overlap_train_test = train.index.intersection(test.index)
overlap_val_test = validation.index.intersection(test.index)

print("2. train-validation overlap:", len(overlap_train_val))
print("   train-test overlap:", len(overlap_train_test))
print("   validation-test overlap:", len(overlap_val_test))

# 3. Row count conservation
total_check = len(train) + len(validation) + len(test) == len(merged)
print("3. Row count conserved:", total_check)

# 4. Minimum train coverage per user
users_in_merged = set(merged['user_id'].unique())
users_in_train = set(train['user_id'].unique())
missing_users = users_in_merged - users_in_train
print("4. Users with zero rows in train:", len(missing_users))

# 5. User count consistency
n_users_merged = merged['user_id'].nunique()
n_users_test = test['user_id'].nunique()
n_users_val = validation['user_id'].nunique()
print("5. User counts -> merged:", n_users_merged, "| test:", n_users_test, "| validation:", n_users_val)

1. Temporal leakage violations: 90
2. train-validation overlap: 0
   train-test overlap: 0
   validation-test overlap: 0
3. Row count conserved: True
4. Users with zero rows in train: 0
5. User counts -> merged: 657203 | test: 657203 | validation: 657203


In [18]:
violating_users = leakage_check[~leakage_check].index

sample_user = violating_users[0] ## Pick the first violating user as a sample
print("Sample violating user:", sample_user)
print(merged[merged['user_id'] == sample_user][['user_id', 'parent_asin', 'rating', 'timestamp']])

Sample violating user: AE35XTX3NJDSLVQLP535XFDP5XYQ
                              user_id parent_asin  rating      timestamp
6560124  AE35XTX3NJDSLVQLP535XFDP5XYQ  B0040QYROA     1.0  1236536590000
6560125  AE35XTX3NJDSLVQLP535XFDP5XYQ  B0000A0MFJ     4.0  1236536993000
6560126  AE35XTX3NJDSLVQLP535XFDP5XYQ  B001KVZ6FW     5.0  1263054436000
6560127  AE35XTX3NJDSLVQLP535XFDP5XYQ  B005OCFGTO     5.0  1426463768000
6560128  AE35XTX3NJDSLVQLP535XFDP5XYQ  0767827724     5.0  1426463768000
6560129  AE35XTX3NJDSLVQLP535XFDP5XYQ  B000654ZK0     5.0  1552332896762


**Why 90 "leakage" flags appeared**

A sample user has two rows with the **exact same timestamp**:

| index | parent_asin | rating | timestamp |
|---|---|---|---|
| 6560127 | B005OCFGT0 | 5.0 | 1426463768000 |
| 6560128 | 0767827724 | 5.0 | 1426463768000 |

Since `.tail(1)` can't tell which same-timestamp row is "truly last," our strict 
`<` check flags these as violations — even though nothing is actually leaking.

**Fix:** use `<=` instead of `<` to allow equal timestamps.

In [19]:
leakage_check_fixed = (train_max_ts <= val_ts) & (val_ts <= test_ts)
print("Temporal leakage violations (corrected):", (~leakage_check_fixed).sum())

Temporal leakage violations (corrected): 0


**Result after the fix**

Using `<=` instead of `<`, the corrected check returns:

`Temporal leakage violations (corrected): 0`

This confirms the split has **no real temporal leakage** — the original 90 flags 
were only caused by users with duplicate (equal) timestamps, not by any actual 
ordering error.